## Llama Inferenz Server (nur Intel)

Beispiel für einen Client, der mit einem Llama-Inference-Server auf einem separaten Rechner kommuniziert.

Auf diesem Server läuft der Llama-Inference-Service als zentral betriebener Dienst und stellt das geladene Modell über eine OpenAI-kompatible API für mehrere Clients bereit. Der Server ist Teil des Ollama-Ökosystems und ermöglicht es, Modelle zentral zu hosten und über standardisierte Schnittstellen anzusprechen. 

Im Unterschied zu anderen Servern ist die Integration stärker auf lokale und einfach deploybare Setups ausgelegt, wobei typischerweise mehrere Modelle innerhalb derselben Laufzeitumgebung verwaltet werden können.

Die Verbindung erfolgt über den API-Endpoint des Servers. Der verwendete API-Key dient dabei lediglich als Platzhalter (je nach Konfiguration wird bei Ollama kein echter API-Key benötigt).

Nach dem Setzen der Umgebungsvariablen deployen wir den Inference-Server mittels Helm.

In [ ]:
%%bash
source ~/data/env.py
cat <<EOF | tee ~/data/env.py
OPENAI_API_KEY="$OPENAI_API_KEY"
HF_TOKEN=""
AI_KUBECONFIG="$AI_KUBECONFIG"
AI_MODEL="HuggingFaceTB/SmolLM2-1.7B-Instruct-GGUF:Q4_K_M"
AI_NAME="smollm2-gguf"
AI_IP="${AI_IP}"
EOF

In [ ]:
%%bash
source ~/data/env.py
helm ${AI_KUBECONFIG} upgrade --install ${AI_NAME} helm/llama/  --set server.hfModel="${AI_MODEL}"

Bevor wir den einfachen Test starten können, muss der Inferenz Server bereit sein:

In [ ]:
%%bash
source ~/data/env.py
kubectl ${AI_KUBECONFIG} get pods,services
kubectl ${AI_KUBECONFIG} logs deployment/${AI_NAME}-llama-server --tail=10 || true

### Testen

Vorher muss base_url gesetzt werden.

In [ ]:
%%bash
source ~/data/env.py
echo "AI_BASE_URL=\"http://${AI_IP}:$(kubectl $AI_KUBECONFIG get service ${AI_NAME}-llama-server -o jsonpath='{.spec.ports[0].nodePort}')/v1\"" | tee -a ~/data/env.py

---

## Weitere LLMs 

Deployt weitere LLMs und versucht obigen Prompt, Analysiert die Ausgabe.

Mögliche LLMs sind:
* **HuggingFaceTB/SmolLM2-135M-Instruct** - sehr kleines Modell neigt zu Halluzinationen
* **Qwen2.5-Coder-0.5B** - grösser, gut zum Entwicklern von Software
* **TinyLlama-1.1B-Chat-v1.0** - ebenfalls grösser, für allgemeine Textabfragen
* **Qwen2.5-7B-Instruct** - für allgemeine Textabfragen bei Helm muss zusätzlich `--set model.memFraction=0.40` mitgegeben werden.
* **swiss-ai/Apertus-8B-Instruct-2509** - Schweizer LLM, komplett OpenSource. Bei Helm muss zusätzlich `--set model.memFraction=0.40` mitgegeben werden.

**Hinweis**: neben dem Modell muss ich der Modellname angepasst werden.

---
### Weitere Beispiele

Wechselt in das Verzeichnis `03-openai` und spielt die Übungen durch. 

Welche Funktionieren und welche nicht?

---
### Aufräumen

In [ ]:
%%bash
source ~/data/env.py
helm ${AI_KUBECONFIG} uninstall ${AI_NAME} || true